In [1]:
import pandas as pd

## 1. Comienzo por filtrar el dataset:

### 1.1 Primero saco las muestras de paises non-US:

In [ ]:
chunks = pd.read_json("goodreads_books.json", lines=True, chunksize=100_000)

us_list = []
for chunk in chunks:
    us_list.append(chunk[chunk["country_code"] == "US"])

df_us = pd.concat(us_list, ignore_index=True)
print(df_us.shape)


### 1.2 Ahora me tengo que quedar nomas con los libros que son en ingles:

Consideramos los libros en ingles los que tienen "languade_code" = ["en", "eng", "en-US", "en-GB", "en-CA", "enm", "ang", "sco", "gem"]

Las ultimas 4 opciones son ingles, pero antiguos (afectara en el rating del libro?... Acaso como a las personas de hoy en dia les cosatara mas atenderlo y por eso le daran bajo rating?...)

Tambien me quedo con las muestras con ese valor NaN, porque aun no podemos decidir si descartarlos o no.

In [ ]:
valid_langs = {"en", "eng", "en-us", "en-gb", "en-ca", "enm", "ang", "sco", "gem"}
lang = df_us["language_code"].fillna("").str.strip().str.lower()
df_us_eng = df_us[lang.isin(valid_langs) | (lang == "")]
print(df_us_eng.shape)

Lo guardo en un json:

In [ ]:
import json
output_path = "goodreads_us_english.json"
chunk_size = 10000
with open(output_path, "w", encoding="utf-8") as f:
    for start in range(0, len(df_us_eng), chunk_size):
        chunk = df_us_eng.iloc[start:start + chunk_size]
        records = chunk.where(pd.notna(chunk), None).to_dict(orient="records")
        for record in records:
            json.dump(record, f, ensure_ascii=False)
            f.write("\n")
print("Guardado en", output_path)

Ahora abro el nuevo archivo que acabo de crear, asi trabajo sobre ese:

In [2]:
chunks2 = pd.read_json("goodreads_us_english.json", lines=True, chunksize=100_000)

chunk_list = []
for chunk in chunks2:
    chunk_list.append(chunk)

df_us_en = pd.concat(chunk_list, ignore_index=True)
print(df_us_en.shape)

(1925872, 29)


## 2. Reorganizo valores de features:

### 2.1 Comienzo por reagrupar los valores de "format"

In [3]:
# format_uniques = df_us_en["format"].unique()
# for v in format_uniques:
#     print(v)

In [4]:
# df_us_en["format"].value_counts().head(50)

In [5]:
mixed = df_us_en["format"].str.contains(
    "paperback.*ebook|ebook.*paperback|kindle.*paperback|paperback.*kindle",
    case=False,
    na=False
)

print(mixed.sum())

224


In [6]:
def normalize_format(value):
    #Si el valor no es un string:
    if not isinstance(value, str):
        return "UNKNOWN"
    #Si no hay un valor para la muestra o es un espacio en blanco:
    s = value.strip().lower()
    if s == "":
        return "UNKNOWN"
    if "unknown binding" in s or s == "unknown":
        return "UNKNOWN"

    #Reemplazamos &amp; (código HTML para &) por solo &:
    s = s.replace("&amp;", "&")

    #Hacemos unas correcciones de cosas que aparecen frecuentemente en el dataset:
    s = s.replace("papperback", "paperback")
    s = s.replace("papaerback", "paperback")
    s = s.replace("paberback", "paperback")
    s = s.replace("hardcovver", "hardcover")
    s = s.replace("hardover", "hardcover")
    s = s.replace("harcover", "hardcover")


    #Ebook (digital)
    ebook_terms = [
        "ebook", "e-book", "e book", "kindle", "nook", "nookbook", "ibook", "epub", "pdf", "electronic",
        "digital", "online", "wattpad", "webtoon", "webcomic", "web comic", "online fiction", "online fanfiction",
        "free online fiction", "free online read", "online read", "online serial", "online publication", "digital comic",
        "fanfiction"
    ]

    #Formato de audios
    audio_terms = [
        "audio", "audiobook", "audible", "audio cd", "audio cassette", "audio book",
        "audio play", "audio lecture", "audio podcast", "mp3", "cd audio", "cassette",
        "playaway", "digital audio", "overdrive audio"
    ]

    #Hardcover (hardback)
    hardcover_terms = [
        "hardcover", "hard cover", "hardback", "hardbound", "hard bound", "casebound", "library binding",
        "school binding", "cloth bound", "clothbound", "leather bound", "leatherbound", "slipcased",
        "slipcase", "flexbound", "hardcover with slipcase"
    ]

    #Paperback (softcover)
    paperback_terms = [
        "paperback", "softcover", "soft cover", "trade paperback", "mass market paperback", "paper back",
        "paperbound", "pocket paperback", "capa mole", "brochura", "broche", "brochure", "brossura", "paperback with dvd",
        "paperback with flaps", "paperback (large print)", "perfect paperback", "mass paperback", "softback",
        "quality paperback", "paperback manga", "spiral-bound", "spiral bound", "chapbook", "paper", 
    ]

    #Comic / manga / graphic
    comic_terms = ["comic", "graphic novel", "manga", "comic book", "graphic", "manhwa", "manhua", "fancomic"]

    if any(term in s for term in audio_terms):
        return "AUDIO"

    if any(term in s for term in comic_terms):
        return "COMIC"

    if any(term in s for term in hardcover_terms):
        return "HARDCOVER"

    if any(term in s for term in paperback_terms):
        return "PAPERBACK"

    if any(term in s for term in ebook_terms):
        return "EBOOK"

    return "OTHER"

In [7]:
#Le aplico la funcion para reagrupar:
if "format_group" not in df_us_en.columns:
    df_us_en["format_group"] = df_us_en["format"].apply(normalize_format)

print(df_us_en["format_group"].value_counts())

format_group
PAPERBACK    683447
UNKNOWN      591456
EBOOK        293266
HARDCOVER    281921
AUDIO         67250
OTHER          7742
COMIC           790
Name: count, dtype: int64


In [8]:
df_us_en["format_group"].value_counts(normalize=True) * 100

format_group
PAPERBACK    35.487665
UNKNOWN      30.711075
EBOOK        15.227699
HARDCOVER    14.638616
AUDIO         3.491925
OTHER         0.402000
COMIC         0.041020
Name: proportion, dtype: float64

Mas o menos el 99.59% de las muestars quedaron clasificadas en categorias concretas --> esto esta muy bueno!
Y podemos ver que hay muhcos UNKOWNS, lo que signfica que el usuario no compleot la info

## 3. Analizo features:

### 3.1 Veo cuantas muestras tienen el laguage_code incompleto:

In [9]:
missing = df_us_en["language_code"].isna().sum() #Aplica para NaN y None
empty_str = (df_us_en["language_code"] == "").sum()
print("NaN:", missing)
print("Cadena vacía:", empty_str)
print("Total vacío:", missing + empty_str)

NaN: 0
Cadena vacía: 1059663
Total vacío: 1059663


### 3.2 ¿El formato del libro tiene alguna relacion con el rating?

In [ ]:
df_us_en.groupby("format_group").apply(lambda x: pd.to_numeric(x["average_rating"], errors="coerce").mean())

In [ ]:
df_us_en.groupby("format_group")["average_rating"].agg(["count","mean","std"])

saga func clari , incluye rautng graf
autores agrupar para luego inflkuye rating graf 